# GANs (Generative Adversarial Networks) (generator vs discriminator)

_Deep Learning_

**A forger and a detective compete; the forger gets so good its fakes look real.**

A GAN (Generative Adversarial Network) makes new, realistic data, like fake photos of people who don't exist.

     It has two networks playing a game. The generator is a forger making fakes. The discriminator is a detective judging real vs fake.

---

This notebook is a practice scaffold for the **GANs (Generative Adversarial Networks) (generator vs discriminator)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

A GAN trains two networks against each other. We'll build both, then run a single round of the game: first update the **discriminator** (the detective) to tell real from fake, then update the **generator** (the forger) to fool it.

### Step 1 — Build the generator, discriminator, and optimizers

The generator `G` turns an 8-dimensional noise vector into a 4-dimensional fake sample. The discriminator `D` takes a 4-dimensional sample and outputs a single score for "how real is this?". We use binary cross-entropy on the logits as the loss, and give each network its own Adam optimizer so they learn independently.

In [ ]:
import torch
import torch.nn as nn

G = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 4))   # noise -> fake
D = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 1))   # sample -> real?
bce = nn.BCEWithLogitsLoss()
optG = torch.optim.Adam(G.parameters(), lr=1e-3)
optD = torch.optim.Adam(D.parameters(), lr=1e-3)

### Step 2 — Draw a batch of real samples and generator noise

We grab a batch of 32 "real" samples (random stand-ins for a real dataset) and a batch of 32 noise vectors that seed the generator. The noise is the raw material the forger shapes into fakes.

In [ ]:
real = torch.randn(32, 4)               # a batch of real samples
noise = torch.randn(32, 8)              # random seed for the generator

### Step 3 — Train the discriminator

The detective learns to score real samples toward 1 and fakes toward 0. We generate fakes with the current generator and `detach()` them so the generator's weights are **not** updated on this step — only the discriminator learns here. Its loss is the BCE on reals (target 1) plus the BCE on fakes (target 0).

In [ ]:
# 1) train discriminator: real -> 1, fake -> 0
fake = G(noise).detach()
optD.zero_grad()
lossD = bce(D(real), torch.ones(32, 1)) + bce(D(fake), torch.zeros(32, 1))
lossD.backward()
optD.step()

### Step 4 — Train the generator

Now the forger gets its turn. We pass fresh fakes through the discriminator but label them as **real** (target 1): the generator's loss is low when the discriminator is fooled. Backpropagating this loss nudges the generator toward producing samples the detective accepts.

In [ ]:
# 2) train generator: fool D into calling fakes real (label 1)
optG.zero_grad()
lossG = bce(D(G(noise)), torch.ones(32, 1))
lossG.backward()
optG.step()
print("lossD", round(lossD.item(), 3), "lossG", round(lossG.item(), 3))

## Visualize the data & results

_As a generator learns to mimic real digit-pixel statistics, does its output distance shrink toward the real data?_

### Step 1 — Measure the real data's target statistic

A full GAN matches an entire distribution; here we illustrate the idea with a single statistic — the average pixel brightness of the real digit images. The generator's goal is to make its own output match this number.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
real_mean = (digits.data / 16.0).mean()  # real target statistic = 0.263

### Step 2 — Simulate the generator closing the gap

The generator starts producing dark images (mean 0) and, each step, moves a fraction of the way toward the real mean — a simple stand-in for what gradient updates do over training. We record the value at every step so we can plot the trajectory.

In [ ]:
fake = 0.0                               # generator starts producing dark images
trace = []
for step in range(41):
    trace.append(fake)
    fake += 0.1 * (real_mean - fake)     # learns toward the real data statistic

### Step 3 — Plot the convergence

The blue curve is the generator's mean pixel value over training; the green line is the real target. As the forger improves, the gap closes and the two lines meet — the visual signature of a generator learning to mimic real statistics.

In [ ]:
plt.plot(trace, color="#4ea1ff", label="mean pixel of fake batch")
plt.axhline(real_mean, color="#7ee787", label="real digit mean pixel (target)")
plt.title("Generator-vs-real distance and per-pixel mean error on load_digits")
plt.xlabel("step"); plt.ylabel("distance / error")
plt.legend()
plt.show()